In [1]:
# imports 
import numpy as np
from tqdm import tqdm
import pandas as pd
import numpy as np
import pickle
from scipy.optimize import minimize
from scipy.stats import beta as beta_dist

npa= np.array

In [2]:
#define parameters bounds 
alpha_bnd = (0.001,0.999) #learning rate
beta_bnd = (0.001, 99.99) #inverse temperature
rho_bnd = (0.001,1.999) #threshold (only for foraging model)

kappa_bnd = (0.001,9.999) #for DBM prior
p0_bnd = (0.001,0.999) #for DBM prior

C_bnd = (0.001,9.999) # ind term EXP
eta_bnd = (-0.999,0.999) #input term EXP

epsilon = 1e-10 #to avoid log underflow 

# Load and preprocess data

In [ ]:
#download data sample named "Experiment1_data" and change dataPath to your local path 
dataPath = "Experiment1_data.pickle" #"data/Experiment1_data.pickle" #

#extract data
with open(dataPath, 'rb') as handle:
    all_sub_2ab = pickle.load(handle)

subs = all_sub_2ab.keys()

excluded_subs = []

for sub in tqdm(subs) : 
    df= all_sub_2ab[sub]
    df= df.drop(df.index[0:25])
    switch = np.array((df['choice'] != df['choice'].shift()).cumsum())[-1]

    #exclude partcipants that didn't switch at least once
    if switch < 2 : 
        excluded_subs.append(sub) 
    
subs = [sub for sub in subs if sub not in excluded_subs]

/var/folders/20/11y1vnk95q39s6c8yyyx446m0000gn/T/ipykernel_76650/2329542087.py:6: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  all_sub_2ab = pickle.load(handle)
100%|██████████| 258/258 [00:00<00:00, 3820.85it/s]


In [7]:
def LSE(x): 
    x = np.array(x)
    c=np.max(x)
    return c + np.log(np.sum(np.exp(x - c)))

# Models

## Compare alternatives

In [6]:
def ql_model(theta,df):

    sideList= df['choice'].tolist()
    rewardList = df['reward'].tolist()
    
    karm= max(sideList)+1
    Qvalue = np.zeros(karm)#+(1/karm) 

    alpha = theta[0]
    beta = theta[1]

    probList=[]

    for i in range (len(rewardList)):
        arm = int(sideList[i]) 

        
        ##Softmax function
        prob = np.exp((Qvalue[arm]*beta)-LSE(Qvalue*beta))
        
        probList.append(prob)
        Qvalue [arm] += alpha*(rewardList[i]-Qvalue[arm])

        ##log likelihood
    logLike = (np.sum(np.log(npa(probList)+epsilon))) * (-1)
    
    return logLike

In [7]:
def ql_model_ab(theta,df):

    sideList= df['choice'].tolist()
    rewardList = df['reward'].tolist()
    
    karm= max(sideList)+1
    Qvalue = np.zeros(karm)#+(1/karm) 

    alpha = theta[0]
    beta0 = theta[1]
    alpha_rho = theta[2]
    beta1 = theta[3]

    mu = 0 #rho_0

    probList=[]

    for i in range (len(rewardList)):
        arm = int(sideList[i]) 

        beta = beta0 + beta1*mu
        ##Softmax function
        prob = np.exp((Qvalue[arm]*beta)-LSE(Qvalue*beta))
        for a in range(karm):
            if a == arm:
                Qvalue[a] += alpha*(rewardList[i]-Qvalue[a])
            else:
                Qvalue[a] += 0 
        probList.append(prob)
        
        mu += alpha_rho*(rewardList[i]- mu)

        ##log likelihood
    logLike = (np.sum(np.log(npa(probList)+epsilon))) * (-1)
    
    return logLike

In [8]:
def ql_model_abck(theta,df):

    sideList= df['choice'].tolist()
    rewardList = df['reward'].tolist()
    
    karm= max(sideList)+1
    Qvalue = np.zeros(karm) #+(1/karm) 
    CKvalue = np.zeros(karm)
    
    alpha = theta[0]
    beta0 = theta[1]
    alpha_rho = theta[2]
    beta1 = theta[3]
    alpha_ck = theta[4]
    beta_ck = theta[5]

    mu = 0 

    probList=[]

    for i in range (len(rewardList)):
        arm = int(sideList[i]) 

        ##Softmax function
        beta = beta0 + beta1*mu
        prob = np.exp((Qvalue[arm]*beta + CKvalue[arm]*beta_ck)-LSE(Qvalue*beta + CKvalue*beta_ck))
        for a in range(karm):
            if a == arm:
                CKvalue[arm] += alpha_ck*(1-CKvalue[arm])
                Qvalue[a] += alpha*(rewardList[i]-Qvalue[a])
            else:
                CKvalue[a] += alpha_ck*(0-CKvalue[arm])
                Qvalue[a] += 0 #gamma*(rho-Qvalue[a]) #gamma*rho #
        probList.append(prob)
        
        mu += alpha_rho*(rewardList[i]- mu)

        ##log likelihood
    logLike = (np.sum(np.log(npa(probList)+epsilon))) * (-1)
    
    return logLike

## Compare to threshold

In [9]:
def forage_model(theta,df):  
    
    alpha = theta[0]
    beta = theta[1]   
    rho = theta[2]
    
    sideList= df['choice'].tolist()
    rewardList = df['reward'].tolist()
    karm= max(sideList)+1
   
    oitOreList= []

    #switch/explore = 0, stay/exploit = 1
    for j in range (len(sideList)-1):  
        if sideList[j]==sideList[j+1]:
            oitOreList.append(1)
        else:
            oitOreList.append(0)


    v_oit = 1 #rho

    probList= [1/karm]
    v_oit += alpha*(rewardList[0]-v_oit)

    for i in range (1,len(rewardList)):
        oit = oitOreList[i-1] # oit or ore 

        if oit == 1:
            prob = np.exp((v_oit*beta)-LSE([v_oit*beta, rho*beta]))
            probList.append(prob)
            v_oit += alpha*(rewardList[i]-v_oit)

        if oit == 0:
            prob = 1-(np.exp((v_oit*beta)-LSE([v_oit*beta, rho*beta])))
            probList.append(prob)
            v_oit = rho + alpha*(rewardList[i]-rho) 
    

    ##log likelihood
    logLike = (np.sum(np.log(npa(probList)+epsilon))) * (-1)
  
    return logLike

In [10]:
def forage_model_ab(theta,df):  

    alpha = theta[0]
    beta0 = theta[1]   
    alpha_rho = theta[2]
    beta1= theta[3]
    rho = theta[4]
    
    sideList= df['choice'].tolist()
    rewardList = df['reward'].tolist()
    karm= max(sideList)+1
   
    oitOreList= []

    #switch/explore = 0, stay/exploit = 1
    for j in range (len(sideList)-1):  
        if sideList[j]==sideList[j+1]:
            oitOreList.append(1)
        else:
            oitOreList.append(0)
    #initial v(exploit)
    v_oit = 1 #rho_0
    #initial richness
    mu = 0
    #likelihood for first observation
    probList= [1/karm]

    v_oit += alpha*(rewardList[0]-v_oit)
    mu += alpha_rho*(rewardList[0]- mu)
    
    for i in range (1,len(rewardList)):
        oit = oitOreList[i-1] # oit or ore 
        
        beta = beta0 + beta1*mu
        if oit == 1:
            prob = np.exp((v_oit*beta)-LSE([v_oit*beta, rho*beta]))
            probList.append(prob)
            v_oit += alpha*(rewardList[i]-v_oit)

        if oit == 0:
            prob = 1-(np.exp((v_oit*beta)-LSE([v_oit*beta, rho*beta])))
            probList.append(prob)
            v_oit = rho + alpha*(rewardList[i]-rho) 
        #track richness
        mu += alpha_rho*(rewardList[i]- mu)

    ##log likelihood
    logLike = (np.sum(np.log(npa(probList)+epsilon))) * (-1)
  
    return logLike

In [11]:
def forage_model_abck(theta,df):  
    
    alpha = theta[0]
    beta0 = theta[1]   
    alpha_rho = theta[2]
    beta1 = theta[3]
    rho = theta[4]
    alpha_ck = theta[5]
    beta_ck = theta[6]
    
    sideList= df['choice'].tolist()
    rewardList = df['reward'].tolist()
    karm= max(sideList)+1
   
    oitOreList= []

    #switch/explore = 0, stay/exploit = 1
    for j in range (len(sideList)-1):  
        if sideList[j]==sideList[j+1]:
            oitOreList.append(1)
        else:
            oitOreList.append(0)

    v_oit = 1
    mu = 0
    CKvalue = 0
    probList= [1/karm]

    v_oit += alpha*(rewardList[0]-v_oit)
    mu += alpha_rho*(rewardList[0]- mu)

    for i in range (1,len(rewardList)):
        oit = oitOreList[i-1] # oit or ore 
        
        beta = beta0 + beta1*mu
        if oit == 1:
            prob = np.exp((v_oit*beta + CKvalue*beta_ck)-LSE([v_oit*beta + CKvalue*beta_ck, rho*beta]))
            probList.append(prob)
            CKvalue += alpha_ck*(1-CKvalue)
            v_oit += alpha*(rewardList[i]-v_oit)

        if oit == 0:
            prob = 1-(np.exp((v_oit*beta + CKvalue*beta_ck)-LSE([v_oit*beta + CKvalue*beta_ck, rho*beta])))
            probList.append(prob)
            CKvalue = 0
            v_oit = rho + alpha*(rewardList[i]-rho) 
        #track richness
        mu += alpha_rho*(rewardList[i]- mu)
        

    ##log likelihood
    logLike = (np.sum(np.log(npa(probList)+epsilon))) * (-1)
  
    return logLike

## Dynamic Belief Model

In [18]:
def dbm_fixedprior(theta,df, resolution=50): ##fixed p0 and kappa, not used in paper, but included for completeness.
    """
    Calculates the Negative Log Likelihood of MAB data under the Dynamic Belief Model.
    
    Args:
        theta (params) (list/tuple): [hazard_rate, inverse_temp]
                             hazard_rate: prob of change (0 to 1)
                             inverse_temp: softmax beta parameter (> 0)
        df that contains:
        - choices (array-like): Array of integers representing arm choices (0, 1, 2...)
        - rewards (array-like): Array of binary rewards (0 or 1) corresponding to the choices.
        resolution (int): Grid resolution for the probability distribution.
        
    Returns:
        float: Negative Log Likelihood (scalar).
    """
    
    # 1. Unpack Parameters
    hazard_rate = theta[0]
    beta = theta[1]
    kappa = theta[2]
    p0 = theta[3]
    
    # Sanity check bounds (helps optimizers that don't support bounds strictly)
    if hazard_rate < 0.0001 or hazard_rate > 0.9999: return 1e9
    if beta < 0: return 1e9

    choices= df['choice'].tolist()
    rewards = df['reward'].tolist()

    choices = np.array(choices).astype(int)
    rewards = np.array(rewards)
    n_trials = len(choices)
    n_arms = np.max(choices) + 1
    
    # 2. Initialize Grid and Priors
    # Grid: values from 0 to 1
    grid = np.linspace(0, 1, resolution)
    a = kappa*p0
    b = kappa*(1-p0)
    # Prior: Beta distribution (Beta(a,b))
    prior_pdf = beta_dist.pdf(grid, a, b)
    prior_pdf /= np.sum(prior_pdf) # Normalize
    
    # Posterior Matrix: [n_arms, resolution]
    # Initialize all arms to the prior
    posteriors = np.tile(prior_pdf, (n_arms, 1))
    
    # Variable to accumulate Log Likelihood
    nll = 0.0
    
    # Tiny epsilon to prevent log(0)
    epsilon = 1e-10

    # 3. Loop through trials
    for t in range(n_trials):
        
        # --- A. Calculate Expectations (Mean of Posterior) ---
        # This computes the dot product of the grid and probability for all arms at once
        # expected_values shape: [n_arms]
        expected_values = np.sum(posteriors * grid, axis=1)
        
        # --- B. Softmax Choice Rule ---
        # Q values are expected values scaled by beta
        Q = beta * expected_values
        
        # Numerically stable softmax
        Q_max = np.max(Q)
        exp_Q = np.exp(Q - Q_max)
        probs_choice = exp_Q / np.sum(exp_Q)
        
        # --- C. Accumulate Likelihood ---
        # The prob assigned to the arm actually chosen
        chosen_arm = choices[t]
        observed_reward = rewards[t]
        
        prob_taken = probs_choice[chosen_arm]
        nll -= np.log(prob_taken + epsilon)
        
        # --- D. Update Beliefs (The DBM Step) ---
        
        # 1. PREDICTION STEP (Applies ONLY to the CHOSEN arm)
        # We only mix the prior into the chosen arm's belief.
        posteriors[chosen_arm, :] = (1 - hazard_rate) * posteriors[chosen_arm, :] + hazard_rate * prior_pdf
        
        # 2. UPDATE STEP (Applies ONLY to the CHOSEN arm)
        # We multiply by the likelihood of the reward observed
        if observed_reward == 1:
            likelihood = grid # The probability of getting a 1 is gamma
        else:
            likelihood = 1.0 - grid # The probability of getting a 0 is (1-gamma)
            
        # Apply likelihood only to the chosen arm row
        posteriors[chosen_arm, :] *= likelihood
        
        # 3. NORMALIZE (Only needed for chosen arm, but safer to do all or just chosen)
        # The unchosen arms are already normalized because (1-H)*sum(1) + H*sum(1) = 1
        # So we only strictly need to normalize the chosen one.
        norm_factor = np.sum(posteriors[chosen_arm, :])
        
        if norm_factor > 0:
            posteriors[chosen_arm, :] /= norm_factor
        else:
            # Fallback for numerical underflow (rare)
            posteriors[chosen_arm, :] = prior_pdf

    return nll

In [19]:
def dbm_fixedkappa(theta, df, kappa = 20, resolution=50):
    """
    DBM with fixed concentration (kappa). 
    theta now contains: [hazard_rate, beta, mean_prior]
    where mean_prior is the mean of Beta(a,b), and we compute a,b from kappa.
    """
    hazard_rate = theta[0]
    beta = theta[1]
    mean_prior = theta[2]
    
    # Sanity checks
    if hazard_rate < 0.0001 or hazard_rate > 0.9999: return 1e9
    if beta < 0: return 1e9
    if mean_prior < 0.001 or mean_prior > 0.999: return 1e9
    
    # Derive a, b from mean and kappa
    # mean = a / (a + b), so a = mean * kappa, b = (1-mean) * kappa
    a = mean_prior * kappa
    b = (1.0 - mean_prior) * kappa
    
    choices = df['choice'].tolist()
    rewards = df['reward'].tolist()
    
    choices = np.array(choices).astype(int)
    rewards = np.array(rewards)
    n_trials = len(choices)
    n_arms = np.max(choices) + 1
    
    # Grid and prior
    grid = np.linspace(0, 1, resolution)
    prior_pdf = beta_dist.pdf(grid, a, b)
    prior_pdf /= np.sum(prior_pdf)
    
    posteriors = np.tile(prior_pdf, (n_arms, 1))
    nll = 0.0
    epsilon = 1e-10
    
    for t in range(n_trials):
        expected_values = np.sum(posteriors * grid, axis=1)
        Q = beta * expected_values
        Q_max = np.max(Q)
        exp_Q = np.exp(Q - Q_max)
        probs_choice = exp_Q / np.sum(exp_Q)
        
        chosen_arm = choices[t]
        observed_reward = rewards[t]
        prob_taken = probs_choice[chosen_arm]
        nll -= np.log(prob_taken + epsilon)
        
        posteriors[chosen_arm, :] = (1 - hazard_rate) * posteriors[chosen_arm, :] + hazard_rate * prior_pdf
        
        if observed_reward == 1:
            likelihood = grid
        else:
            likelihood = 1.0 - grid
        
        posteriors[chosen_arm, :] *= likelihood
        norm_factor = np.sum(posteriors[chosen_arm, :])
        
        if norm_factor > 0:
            posteriors[chosen_arm, :] /= norm_factor
        else:
            posteriors[chosen_arm, :] = prior_pdf
    
    return nll

In [17]:
def exp_dbm_fixedprior(theta,df): #### fixed p0 and kappa, not used in paper, but included for completeness.

    sideList= df['choice'].tolist()
    rewardList = df['reward'].tolist()
    
    #constant parameters for exp-dbm
    p0 = 0.5
    G = 1/3

    karm= max(sideList)+1
    Qvalue = np.zeros(karm)#+(1/karm) 

    alpha = theta[0]
    beta = theta[1]

    probList=[]

    for i in range (len(rewardList)):
        arm = int(sideList[i]) 

        
        ##Softmax function
        prob = np.exp((Qvalue[arm]*beta)-LSE(Qvalue*beta))
        
        probList.append(prob)
        Qvalue [arm] = (1-alpha)*p0 + alpha*(G*rewardList[i] + Qvalue[arm]*(1-G))

        ##log likelihood
    logLike = (np.sum(np.log(npa(probList)+epsilon))) * (-1)
    
    return logLike

In [10]:
def exp_dbm_fixedkappa(theta,df, kappa = 4):

    sideList= df['choice'].tolist()
    rewardList = df['reward'].tolist()
    
    karm= max(sideList)+1
    Qvalue = np.zeros(karm)#+(1/karm) 

    alpha = theta[0]
    beta = theta[1]
    p0 = theta[2]

    a = p0 * kappa
    b = (1.0 - p0) * kappa

    G = 1/(a + b + 1)
    probList=[]

    for i in range (len(rewardList)):
        arm = int(sideList[i]) 

        
        ##Softmax function
        prob = np.exp((Qvalue[arm]*beta)-LSE(Qvalue*beta))
        
        probList.append(prob)
        Qvalue [arm] = (1-alpha)*p0 + alpha*(G*rewardList[i] + Qvalue[arm]*(1-G))

        ##log likelihood
    logLike = (np.sum(np.log(npa(probList)+epsilon))) * (-1)
    
    return logLike

In [15]:
def exp_free(theta,df):

    sideList= df['choice'].tolist()
    rewardList = df['reward'].tolist()
    

    karm= max(sideList)+1
    Qvalue = np.zeros(karm)#+(1/karm) 

    C = theta[0]
    eta = theta[1]
    alpha = theta[2] #as in Angela Yu's paper, but they call it beta
    beta = theta[3]

    probList=[]

    for i in range (len(rewardList)):
        arm = int(sideList[i]) 

        
        ##Softmax function
        prob = np.exp((Qvalue[arm]*beta)-LSE(Qvalue*beta))
        
        probList.append(prob)

        ##
        Qvalue[arm] = C*(1-alpha) + eta*alpha*rewardList[i] + alpha*Qvalue[arm]

        ##log likelihood
    logLike = (np.sum(np.log(npa(probList)+epsilon))) * (-1)
    
    return logLike

# Optimisation 

In [12]:
def fit_model(model, bounds, participant_data, sub, paramString, method = 'L-BFGS-B'):    

    # Number of random starting points to try for each participant
    n_starts = 20 
    num_params = len(bounds)

    
    # Variables to store the best result *for this participant*
    best_nll = np.inf
    best_params_for_p = None
    succ = False
    msg = ""
    options_method = None
    if method == 'Nelder-Mead':
        options_method = {'maxfev': 2000,'maxiter': 2000, 'xatol':0.001, 'fatol': 0.001}
    elif method == 'L-BFGS-B':
        options_method = {'ftol': 1e-6, 'gtol': 1e-6}
    # --- This is the "multiple starts" loop ---
    for j in range(n_starts):
        
        x0 = np.zeros(num_params)
        # Generate random initial guesses (x0) within the bounds
        for n_var in range(num_params):
            x0[n_var] = np.random.uniform(bounds[n_var][0],bounds[n_var][1])
        
        # We need to pass the participant's data to the NLL function.
        # We do this using the `args` argument.
        args_to_pass = participant_data 

        # Run the optimizer!
        result = minimize(
            model,
            x0,
            args=args_to_pass,
            method=method,
            bounds=bounds,
            options = options_method
        )
        
        # Check if this run was successful and if it's the new best
        if result.success and result.fun < best_nll:
            best_nll = result.fun
            best_params_for_p = result.x
            succ = result.success
            msg = result.message
            
    # After 10 starts, store the best parameters we found

    #print("Optimization complete.")
    dataDict = {key: [value] for key, value in zip(paramString, best_params_for_p)}
    dataDict['sub'] = sub
    dataDict['likelihood'] = best_nll
    dataDict['succ'] = succ
    dataDict['msg'] = msg
    opt = pd.DataFrame.from_dict(dataDict)
    return opt




# Fit models

In [ ]:
opt_QL = pd.DataFrame()
opt_QL_ab = pd.DataFrame()
opt_QL_abck = pd.DataFrame()
opt_FOR = pd.DataFrame()
opt_FOR_ab = pd.DataFrame()
opt_FOR_abck = pd.DataFrame()
opt_DBM = pd.DataFrame()
opt_EXP_DBM = pd.DataFrame()
opt_EXP_free = pd.DataFrame()


methodExts = 'Nelder-Mead'
for sub in tqdm(subs) : 
    df= all_sub_2ab[sub]
    #Drop practice trials
    df= df.drop(df.index[0:25])

    # # # Vanilla
    # paramString = ['alpha','beta']
    # bounds = [alpha_bnd,beta_bnd]
    # data = fit_model(ql_model,bounds,df,sub,paramString,method = methodExts)
    # opt_QL = pd.concat([opt_QL,data], axis= 0)
    # paramString = ['alpha','beta','rho_0']
    # bounds = [alpha_bnd,beta_bnd,rho_bnd]
    # data = fit_model(forage_model,bounds,df,sub,paramString,method = methodExts)
    # opt_FOR = pd.concat([opt_FOR,data], axis= 0)

    # # # Adaptive Beta
    # paramString = ['alpha','beta','alpha_rho','beta1']
    # bounds = [alpha_bnd,beta_bnd,alpha_bnd,beta_bnd]
    # data = fit_model(ql_model_ab,bounds,df,sub,paramString,method = methodExts)
    # opt_QL_ab = pd.concat([opt_QL_ab,data], axis = 0)
    # paramString = ['alpha','beta','alpha_rho','beta1','rho_0']
    # bounds = [alpha_bnd,beta_bnd,alpha_bnd,beta_bnd,rho_bnd]
    # data = fit_model(forage_model_ab,bounds,df,sub,paramString,method = methodExts)
    # opt_FOR_ab = pd.concat([opt_FOR_ab,data], axis = 0)

    # # # ## Adaptive beta with choice kernel
    # paramString = ['alpha','beta','alpha_rho','beta1','alpha_ck','beta_ck']
    # bounds = [alpha_bnd,beta_bnd,alpha_bnd,beta_bnd,alpha_bnd,beta_bnd]
    # data = fit_model(ql_model_abck,bounds,df,sub,paramString, method = methodExts)
    # opt_QL_abck = pd.concat([opt_QL_abck,data], axis = 0)    
    # paramString = ['alpha','beta','alpha_rho','beta1','rho_0','alpha_ck','beta_ck']
    # bounds = [alpha_bnd,beta_bnd,alpha_bnd,beta_bnd,rho_bnd,alpha_bnd,beta_bnd]
    # data = fit_model(forage_model_abck,bounds,df,sub,paramString, method = methodExts)
    # opt_FOR_abck = pd.concat([opt_FOR_abck,data], axis = 0)    

    ## Dynamic Belief Model
    # Full p0
    # paramString = ['alpha','beta','p0']
    # bounds = [alpha_bnd,beta_bnd,p0_bnd]
    # data = fit_model(dbm_fixedkappa,bounds,df,sub,paramString,method = methodExts)
    # opt_DBM = pd.concat([opt_DBM,data], axis = 0)
    # EXP p0
    paramString = ['alpha','beta','p0']
    bounds = [alpha_bnd,beta_bnd,p0_bnd]
    data = fit_model(exp_dbm_fixedkappa,bounds,df,sub,paramString,method = methodExts)
    opt_EXP_DBM = pd.concat([opt_EXP_DBM,data], axis = 0)   
    # paramString = ['C','eta','alpha','beta']
    # bounds = [C_bnd,eta_bnd,alpha_bnd,beta_bnd]
    # data = fit_model(exp_free,bounds,df,sub,paramString,method = methodExts)
    # opt_EXP_free = pd.concat([opt_EXP_free,data], axis = 0)

100%|██████████| 254/254 [26:06<00:00,  6.17s/it]


In [20]:
#Save 
directory_fits = "fits_richness/"

## Vanilla
#opt_FOR.to_csv(directory_fits+'FORParams.csv')
#opt_QL.to_csv(directory_fits+'RLParams.csv')

# # #Adaptive beta
#opt_FOR_ab.to_csv(directory_fits+'FORabParams.csv')
#opt_QL_ab.to_csv(directory_fits+'RLabParams.csv')

# # adaptive beta + choice kernel
#opt_QL_abck.to_csv(directory_fits+'RLabckParams.csv')
#opt_FOR_abck.to_csv(directory_fits+'FORabckParams.csv')

# # #DBM-related models
#opt_DBM.to_csv(directory_fits+'DBMp0fixedkappaParams.csv')
#opt_EXP_DBM.to_csv(directory_fits+'EXP_DBMp0fixedkappaParams.csv')
# opt_EXP_free.to_csv(directory_fits+'EXP_freeParams.csv')


# Fitting group level kappa

In [ ]:
# Grid search for optimal kappa (CLEAN VERSION)
kappa_candidates = [] # fill to try different kappa values, e.g. [1, 2, 4, 8]
results_kappa_search = {}

for kappa in kappa_candidates:
    print(f"\n--- Fitting with kappa = {kappa} ---")
    
    opt_DBM_kappa = pd.DataFrame()
    paramString = ['hazard_rate', 'beta', 'mean_prior']
    bounds = [(0.001, 0.999), (0.001, 99.99), (0.001, 0.999)]
    
    for sub in tqdm(subs):
        df = all_sub_2ab[sub]
        df = df.drop(df.index[0:25])
        
        best_nll = np.inf
        best_params = None
        
        for j in range(10):  # 10 random starts
            x0 = np.array([
                np.random.uniform(bounds[0][0], bounds[0][1]),
                np.random.uniform(bounds[1][0], bounds[1][1]),
                np.random.uniform(bounds[2][0], bounds[2][1])
            ])
            
            result = minimize(
                exp_dbm_fixedkappa,
                #dbm_fixed_kappa,
                x0,
                args=(df, kappa),
                method='Nelder-Mead',
                bounds=bounds,
                options={'maxfev': 1000,'maxiter': 1000, 'xatol':0.001, 'fatol': 0.01}
            )
            
            if result.fun < best_nll:
                best_nll = result.fun
                best_params = result.x
        
        if best_params is not None:
            dataDict = {paramString[i]: [best_params[i]] for i in range(len(paramString))}
            dataDict['sub'] = sub
            dataDict['likelihood'] = best_nll
            dataDict['kappa'] = kappa
            opt_DBM_kappa = pd.concat([opt_DBM_kappa, pd.DataFrame.from_dict(dataDict)], axis=0)
    
    avg_nll = opt_DBM_kappa['likelihood'].mean() if len(opt_DBM_kappa) > 0 else np.inf
    results_kappa_search[kappa] = avg_nll
    if len(opt_DBM_kappa) > 0:
        print(f"Average NLL across all participants: {avg_nll:.4f}")
    else:
        print(f"Warning: No valid fits found for kappa = {kappa}")

# Find best kappa
best_kappa = min(results_kappa_search, key=results_kappa_search.get)
print(f"\n=== BEST KAPPA: {best_kappa} (avg NLL = {results_kappa_search[best_kappa]:.4f}) ===")
print(f"\nAll results:\n{pd.DataFrame(list(results_kappa_search.items()), columns=['kappa', 'avg_nll'])}")


--- Fitting with kappa = 2 ---


100%|██████████| 254/254 [13:55<00:00,  3.29s/it]


Average NLL across all participants: 98.0825

--- Fitting with kappa = 4 ---


100%|██████████| 254/254 [13:08<00:00,  3.10s/it]


Average NLL across all participants: 97.7002

--- Fitting with kappa = 6 ---


100%|██████████| 254/254 [12:04<00:00,  2.85s/it]


Average NLL across all participants: 97.7445

--- Fitting with kappa = 8 ---


100%|██████████| 254/254 [11:22<00:00,  2.69s/it]

Average NLL across all participants: 98.0156

=== BEST KAPPA: 4 (avg NLL = 97.7002) ===

All results:
   kappa    avg_nll
0      2  98.082547
1      4  97.700159
2      6  97.744467
3      8  98.015590



## Results for dbm
| kappa |   avg_nll|
| --- | --- |
| 10 | 94.724424 |
|     12 | 94.583016 |
|     14 | 94.479749 |
|     16 | 94.444075 |
|     18 | 94.327510 |
|     20 | 94.267313 |
|     25 | 94.448403 |
|     30 | 94.571537 |

## Results for exp_dbm
| kappa |   avg_nll|
| --- | --- |
| 2 | 98.0825 |
| 4 | 97.7002 |
| 6 | 97.7445 |
| 8 | 98.015590 |
| 10  | 98.313497 |
|     15  | 99.419152 |
|     20 | 100.727343 |
|     25 | 102.173814 |
|     30 | 103.583719 |
